In [ ]:
# Celda 1
# Reference views from the 2024 textured mesh of Madrid (Geoportal del Ayuntamiento, "Modelo 3D realista", served as
# public 3D Tiles). Only the tiles over the block are downloaded, at the finest level, and rendered in Blender from the
# same cameras as the model: aerial views to look at the building "live" and compare. The mesh is a visual reference
# only; nothing of it goes into the model. Independent of the other notebooks (needs the .glb of 14 for the comparison).
%cd ..

GPKG = "data/catastro_manzana.gpkg"      # Footprints of the block (01_catastro)
ORIGIN = (440000.0, 4473700.0, 640.0)
MARGIN = 10.0                             # Meters around the block
Z_RANGE = (600.0, 820.0)                  # Ellipsoidal heights searched (m): ground + 50 m geoid + tallest roof
# Web scene of the viewer (VISOR GEOPORTAL 3D, "Modelo 3D realista"): its "Malla 3D 2024" group lists one 3D Tiles
# service per zone of the city; the zone that contains the block is found below.
WEBSCENE = "https://www.arcgis.com/sharing/rest/content/items/d8bcbe7422a048a6837dafad300ef2be/data?f=json"
OUT_DIR = "sources/malla_madrid_2024"     # Tiles (.b3dm), their .glb, and the list
# Flat ground of known height for the vertical offset: floor of the cloister courtyard, 644.14 m on the cloud.
FLOOR_BOX = (-2.5, 8.5, 93.0, 104.0)      # Local xmin, xmax, ymin, ymax
FLOOR_Z = 644.14
MODEL_GLB = "blender/colegio_imperial_lod3.glb"   # 14_materiales
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
# Cameras: from each side at 45 degrees and from above, on the middle of the block; and two close views.
TARGET = (10.0, 115.0, 20.0)
DISTANCE = 150.0
CLOSE = {"cerca_capillas_sur": [(30.0, 85.0, 45.0), (8.0, 126.0, 22.0)],
         "cerca_claustro": [(45.0, 60.0, 50.0), (3.0, 99.0, 8.0)]}
LENS = 35
SIZE = (1600, 1100)
RENDER_PREFIX = "renders/referencia/malla_"

In [ ]:
# Celda 2
import os
import json
import gzip
import math
import struct
import subprocess
import urllib.request
import numpy as np
import geopandas as gpd
from pyproj import Transformer

def fetch(url):
    with urllib.request.urlopen(url) as response:
        data = response.read()
    return gzip.decompress(data) if data[:2] == b"\x1f\x8b" else data

block = gpd.read_file(GPKG, layer="building_parts").total_bounds
xmin, ymin, xmax, ymax = block[0] - MARGIN, block[1] - MARGIN, block[2] + MARGIN, block[3] + MARGIN
to_ecef = Transformer.from_crs(25830, 4978, always_xy=True)
corners = np.array([to_ecef.transform(x, y, z) for x in (xmin, xmax) for y in (ymin, ymax) for z in Z_RANGE])
print("Block with margin:", [round(v, 1) for v in (xmin, ymin, xmax, ymax)])

def zone_urls():
    scene = json.loads(fetch(WEBSCENE))
    urls = []
    def walk(layers):
        for layer in layers:
            if layer.get("layerType") == "IntegratedMesh3DTilesLayer" and layer["url"] not in urls:
                urls.append(layer["url"])
            walk(layer.get("layers", []))
    walk(scene["operationalLayers"])
    return urls

def inside_box(volume, points):
    """True if the points (in the frame of the box) can touch an oriented box of 3D Tiles."""
    b = np.array(volume["box"])
    center, axes = b[:3], b[3:].reshape(3, 3)
    for axis in axes:
        half = np.linalg.norm(axis)
        d = (points - center) @ (axis / half)
        if d.min() > half or d.max() < -half:
            return False
    return True

# Finest tiles over the block: go down every branch that touches it; a tile is kept when none of its children
# touching the block has content (refinement "REPLACE": the children replace the parent).
def finest_tiles(base, root, local):
    leaves = []
    def walk(tile, prefix):
        if not inside_box(tile["boundingVolume"], local):
            return
        uri = tile.get("content", {}).get("uri")
        if uri and uri.endswith(".json"):
            walk(json.loads(fetch(base + prefix + uri))["root"], prefix + os.path.dirname(uri) + "/")
            return
        before = len(leaves)
        for child in tile.get("children", []):
            walk(child, prefix)
        if len(leaves) == before and uri:
            leaves.append((prefix + uri, tile["geometricError"]))
    walk(root["root"], "")
    return leaves

# Zones overlap at their borders, and a zone can cover the block only with coarse tiles: the one with the finest
# tiles (then the most of them) wins.
candidates = []
for url in zone_urls():
    root = json.loads(fetch(url))
    M = np.array(root["root"]["transform"]).reshape(4, 4).T
    local = (np.linalg.inv(M) @ np.c_[corners, np.ones(len(corners))].T).T[:, :3]
    if not inside_box(root["root"]["boundingVolume"], local):
        continue
    base = url.rsplit("/", 1)[0] + "/"
    leaves = finest_tiles(base, root, local)
    if leaves:
        worst = max(e for _, e in leaves)
        print(base.split("/services/")[1], len(leaves), "tiles, coarsest", round(worst, 2), "m")
        candidates.append((worst, -len(leaves), base, M, local, leaves))
worst, _, BASE, M, local, leaves = min(candidates, key=lambda c: c[:2])
print("Zone:", BASE)

In [ ]:
# Celda 3
# Download (only the tiles not already in OUT_DIR) and take the glb out of each b3dm:
os.makedirs(OUT_DIR, exist_ok=True)
glbs, size = [], 0
for path, _ in leaves:
    name = path.replace("/", "_")
    tile_file = os.path.join(OUT_DIR, name)
    if not os.path.exists(tile_file):
        with open(tile_file, "wb") as f:
            f.write(fetch(BASE + path))
    data = open(tile_file, "rb").read()
    size += len(data)
    # b3dm: 28-byte header, feature and batch tables, then a glb:
    _, _, _, ft_json, ft_bin, bt_json, bt_bin = struct.unpack("<4sIIIIII", data[:28])
    assert b"RTC_CENTER" not in data[28:28 + ft_json]
    glb = tile_file.replace(".b3dm", ".glb")
    with open(glb, "wb") as f:
        f.write(data[28 + ft_json + ft_bin + bt_json + bt_bin:])
    glbs.append(os.path.abspath(glb))
print(len(leaves), "tiles,", round(size / 1e6, 1), "MB | geometric error", sorted({round(e, 2) for _, e in leaves}), "m")

In [ ]:
# Celda 4
# Tile frame -> local frame of the project. The tile frame is the glTF content turned z-up (as Blender imports it),
# then the root transform to ECEF; ECEF -> EPSG:25830 is not linear, but over the block an affine fit is enough
# (error printed). Heights stay ellipsoidal here; Blender moves the mesh down by the offset read on FLOOR_BOX.
to_utm = Transformer.from_crs(4978, 25830, always_xy=True)
src, dst = [], []
for x in np.linspace(xmin, xmax, 5):
    for y in np.linspace(ymin, ymax, 5):
        for z in Z_RANGE:
            e = to_ecef.transform(x, y, z)
            src.append((np.linalg.inv(M) @ np.array([*e, 1.0]))[:3])
            dst.append((x - ORIGIN[0], y - ORIGIN[1], z - ORIGIN[2]))
src, dst = np.array(src), np.array(dst)
A = np.c_[src, np.ones(len(src))]
coef, *_ = np.linalg.lstsq(A, dst, rcond=None)
affine = np.eye(4)
affine[:3, :] = coef.T
print("Affine fit: max error", round(float(np.abs(A @ coef - dst).max()), 4), "m")

In [ ]:
# Celda 5
# Render: the mesh, then the model with the same cameras.
views = {}
for name, azimuth in {"norte": 0, "este": 90, "sur": 180, "oeste": 270}.items():
    a = math.radians(azimuth)
    views[name] = [(TARGET[0] + math.sin(a) * DISTANCE, TARGET[1] + math.cos(a) * DISTANCE, TARGET[2] + DISTANCE), TARGET]
views["cenital"] = [(TARGET[0], TARGET[1] - 1.0, TARGET[2] + 230.0), TARGET]
views.update(CLOSE)
os.makedirs(os.path.dirname(RENDER_PREFIX), exist_ok=True)
config = {"glbs": glbs, "matrix": affine.tolist(), "floor_box": FLOOR_BOX, "floor_z": FLOOR_Z - ORIGIN[2],
          "model": os.path.abspath(MODEL_GLB), "views": views, "lens": LENS, "size": SIZE,
          "out": os.path.abspath(RENDER_PREFIX)}
with open(os.path.join(OUT_DIR, "vistas.json"), "w", encoding="utf-8") as f:
    json.dump(config, f)
run = subprocess.run([BLENDER, "-b", "--factory-startup", "--python", "scripts/vistas_malla.py", "--",
                      os.path.join(OUT_DIR, "vistas.json")], capture_output=True, text=True)
for line in run.stdout.splitlines():
    if line.startswith(("Tiles", "Vertical", "Saved")) or "Error" in line:
        print(line)
if run.returncode != 0 or "Saved" not in run.stdout:
    print(run.stdout[-2000:], run.stderr[-2000:])
    raise RuntimeError("Blender failed")

In [ ]:
# Celda 6
# Side by side (mesh | model), one image per view:
from PIL import Image
for name in views:
    left = Image.open(RENDER_PREFIX + name + ".jpg")
    right = Image.open(RENDER_PREFIX + name + "_modelo.jpg")
    both = Image.new("RGB", (left.width + right.width, left.height), "white")
    both.paste(left, (0, 0))
    both.paste(right, (left.width, 0))
    both.save(RENDER_PREFIX + name + "_lado_a_lado.jpg", quality=88)
print("Saved", len(views), "side-by-side images in", os.path.dirname(RENDER_PREFIX))

In [ ]:
# Celda 7
# True orthophoto of the mesh (straight down, north up), in the box and at the 10 cm/px of the 2023 orthophoto:
# 14_materiales reads the colour of each roof on it (tiles or grey). No shadows and no leaning walls.
ORTHO_BBOX = (439958.0, 4473659.0, 440184.0, 4473885.0)
ORTHO_PX = 0.1
ox0, oy0, ox1, oy1 = (ORTHO_BBOX[0] - ORIGIN[0], ORTHO_BBOX[1] - ORIGIN[1], ORTHO_BBOX[2] - ORIGIN[0], ORTHO_BBOX[3] - ORIGIN[1])
center = ((ox0 + ox1) / 2, (oy0 + oy1) / 2)
ortho_config = {**config, "model": None, "views": {"orto_10cm": [(center[0], center[1], 200.0), (center[0], center[1], 0.0), ox1 - ox0]},
                "size": [round((ox1 - ox0) / ORTHO_PX), round((oy1 - oy0) / ORTHO_PX)], "out": os.path.abspath(OUT_DIR) + os.sep + "malla_"}
with open(os.path.join(OUT_DIR, "vistas_orto10.json"), "w", encoding="utf-8") as f:
    json.dump(ortho_config, f)
run = subprocess.run([BLENDER, "-b", "--factory-startup", "--python", "scripts/vistas_malla.py", "--",
                      os.path.join(OUT_DIR, "vistas_orto10.json")], capture_output=True, text=True)
if run.returncode != 0 or "Saved" not in run.stdout:
    print(run.stdout[-2000:], run.stderr[-2000:])
    raise RuntimeError("Blender failed")
print("Orthophoto:", os.path.join(OUT_DIR, "malla_orto_10cm.jpg"))